# 01. Carga de las muestras

Cada muestra de GEO viene en tres archivos generados por Cell Ranger, el software de 10x Genomics que convierte las lecturas de secuenciación en una matriz de conteo. Cell Ranger separa las lecturas por célula con el código de barras que cada gota agrega a su ARN, las alinea al genoma humano (GRCh38) y cuenta las moléculas de cada gen con un segundo código (UMI) que marca cada molécula original, para no contar dos veces las copias de la amplificación.

| Archivo | Contenido |
|---|---|
| `matrix.mtx` | Conteos: una fila por gen y una columna por célula |
| `barcodes.tsv` | Código de barras de cada célula |
| `features.tsv` | Genes, con su código de Ensembl (por ejemplo ENSG00000111640) y su símbolo (por ejemplo GAPDH). Se usa el símbolo |

Las 17 matrices se unen en un solo objeto AnnData, la estructura de Scanpy que guarda la matriz de células por genes junto con tablas de información por célula (muestra, condición) y por gen.

In [1]:
from pathlib import Path

import anndata as ad
import pandas as pd
import scanpy as sc

DIR_MATRICES = Path("../data/raw/matrices")
DIR_PROCESADOS = Path("../data/processed")

In [4]:
filas = []
for archivo in sorted(DIR_MATRICES.glob("*_matrix.mtx.gz")):
    prefijo = archivo.name.replace("matrix.mtx.gz", "")  
    muestra = prefijo.split("_", 1)[1].rstrip("_")         

    if muestra.endswith("_LS"):
        condicion, donante = "LS", muestra[:-3]
    elif muestra.endswith("_NL"):
        condicion, donante = "NL", muestra[:-3]
    else:
        condicion, donante = "Sano", muestra

    filas.append({"prefijo": prefijo, "muestra": muestra,
                  "donante": donante, "condicion": condicion})

muestras = pd.DataFrame(filas)
print(muestras.to_string())
print(muestras["condicion"].value_counts())

                  prefijo    muestra donante condicion
0         GSM9025365_465_        465     465      Sano
1      GSM9025366_486_LS_     486_LS     486        LS
2      GSM9025367_486_NL_     486_NL     486        NL
3         GSM9025368_632_        632     632      Sano
4         GSM9025369_658_        658     658      Sano
5   GSM9025370_BIO023_LS_  BIO023_LS  BIO023        LS
6      GSM9025371_CON161_     CON161  CON161      Sano
7      GSM9025372_CON171_     CON171  CON171      Sano
8      GSM9025373_CON194_     CON194  CON194      Sano
9   GSM9025374_DER114_LS_  DER114_LS  DER114        LS
10  GSM9025375_DER114_NL_  DER114_NL  DER114        NL
11  GSM9025376_DER115_LS_  DER115_LS  DER115        LS
12  GSM9025377_DER115_NL_  DER115_NL  DER115        NL
13  GSM9025378_DER119_LS_  DER119_LS  DER119        LS
14  GSM9025379_DER119_NL_  DER119_NL  DER119        NL
15  GSM9025380_DER120_LS_  DER120_LS  DER120        LS
16  GSM9025381_DER120_NL_  DER120_NL  DER120        NL
condicion


In [3]:
adatas = {}
for _, fila in muestras.iterrows():
    a = sc.read_10x_mtx(DIR_MATRICES, prefix=fila["prefijo"])
    a.obs["donante"] = fila["donante"]
    a.obs["condicion"] = fila["condicion"]
    adatas[fila["muestra"]] = a
    print(f'{fila["muestra"]:>10}: {a.n_obs:>6} células')

       465:    542 células
    486_LS:   1345 células
    486_NL:    200 células
       632:   1106 células
       658:    593 células
 BIO023_LS:   1249 células
    CON161:    975 células
    CON171:   1081 células
    CON194:   1690 células
 DER114_LS:   2701 células
 DER114_NL:   2022 células
 DER115_LS:   3485 células
 DER115_NL:   2483 células
 DER119_LS:   2699 células
 DER119_NL:   1073 células
 DER120_LS:    505 células
 DER120_NL:    997 células


In [5]:
adata = ad.concat(adatas, label="muestra", index_unique="-", merge="same")
adata.var_names_make_unique()

for col in ["muestra", "donante", "condicion"]:
    adata.obs[col] = adata.obs[col].astype("category")

adata

AnnData object with n_obs × n_vars = 24746 × 36601
    obs: 'donante', 'condicion', 'muestra'
    var: 'gene_ids', 'feature_types'

In [6]:
conteo = (adata.obs.groupby(["condicion", "muestra"], observed=True)
          .size().rename("celulas").reset_index())
print(conteo.to_string(index=False))
print(adata.obs["condicion"].value_counts())

condicion   muestra  celulas
       LS    486_LS     1345
       LS BIO023_LS     1249
       LS DER114_LS     2701
       LS DER115_LS     3485
       LS DER119_LS     2699
       LS DER120_LS      505
       NL    486_NL      200
       NL DER114_NL     2022
       NL DER115_NL     2483
       NL DER119_NL     1073
       NL DER120_NL      997
     Sano       465      542
     Sano       632     1106
     Sano       658      593
     Sano    CON161      975
     Sano    CON171     1081
     Sano    CON194     1690
condicion
LS      11984
NL       6775
Sano     5987
Name: count, dtype: int64


In [7]:
DIR_PROCESADOS.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(DIR_PROCESADOS / "01_crudo.h5ad")